In [ ]:
from datetime import timedelta, datetime
# days = 180
interval_time = datetime.now() - timedelta(days=180)


print(interval_time)

2025-10-08 12:53:37.540156


In [2]:
import psycopg
import spacy
import pandas as pd
from database.db_connection import get_connection 
from spacy.matcher import Matcher, PhraseMatcher

from spacy.pipeline import EntityRuler
from collections import Counter
import matplotlib.pyplot as plt
import datetime

# ================ LOAD ARTICLES ================
# LOAD ARTICLES FROM A LONGER TIME THATS WHY YOURE NOT READING EVERTHING!!!
def load_articles(limit = 5000):
    """Loading articles from database"""
    query = """
        SELECT id, source_name, title, content, published_at
        FROM aggregates_trend
        WHERE published_at > NOW() - INTERVAL '365 days'
        ORDER BY published_at DESC
        LIMIT %s
    """

    with get_connection() as conn:
        data_frame = pd.read_sql(query, conn, params = (limit,))

    data_frame['full_text'] = (data_frame['title'] + '-' + data_frame['content'].fillna(''))
    
    return data_frame

data_frame = load_articles()

/var/folders/sc/ncg3j31n7j7fzg03p443n1mh0000gn/T/ipykernel_27081/3440671030.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, conn, params = (limit,))


In [4]:
# Load a pipeline and create the nlp object (needs to be loaded before adding the patterns)
nlp = spacy.load("en_core_web_md")

In [5]:
# Revised Entity Ruler for Fashion Seasons (tried to capture as many edge cases)

# INSIGHTS: This model doesn't get the entities 100% correct. It's unaware of some fashion entities. 
# Needs to be trained to predict better or provide hardcoded matching patterns.

ruler = nlp.add_pipe("entity_ruler", before = "ner")

fashion_season_patterns = [
    # --------------------- Abbreviated Seasons ---------------------
    # One single token, no spaces: "SS26", "AW2026"
    {
        "label": "FASHION_SEASON",
        "pattern": [
            {"LOWER": {"REGEX": r"^(ss|aw|fw)\d{2,4}$"}}
        ]
    }, 
    
    # One single token with slashes, no spaces: "S/S26", "A/W2026"
    {
        "label": "FASHION_SEASON",
        "pattern": [
            {"TEXT": {"REGEX": r"^[SFA]$"}}, # "f"
            {"TEXT": "/"}, # "/"
            {"TEXT": {"REGEX": r"^[SW]\d{2,4}$"}} # "w2026" not too sure about this, need to find a cleaner way
        ]
    },
    
    # Abbreviated season separated by a slash: "S/S 2026", "A/W 2026"
    {
        "label": "FASHION_SEASON",
        "pattern": [
            {"LOWER": {"IN": ["s", "f", "a"]}},
            {"TEXT": {"IN": "/"}},
            {"LOWER": {"IN": ["s", "w"]}},
            {"TEXT": {"REGEX": r"^\d{2,4}$"}}
        ]
    },

    # Abbreviated season no slash: "SS 2026", "AW 2026"
    {
        "label": "FASHION_SEASON",
        "pattern": [
            {"LOWER": {"IN": ["ss", "fw", "aw"]}},
            {"TEXT": {"REGEX": r"^\d{2,4}$"}}
        ]
    },

    # --------------------- Pre Collections ---------------------
    # Pre-collection variations: "Resort 2026", "Cruise 2026"
    {
        "label": "FASHION_SEASON",
        "pattern": [
            {"LOWER": {"IN": ["resort", "cruise"]}},
            {"TEXT": {"REGEX": r"^\d{2,4}$"}}
        ]
    },


    # Pre-Fall single token
    {
        "label": "FASHION_SEASON",
        "pattern": [
            {"LOWER": "pre-fall"},
            {"TEXT": {"REGEX": r"^\d{2,4}$"}}
        ]
    },

    # Pre-fall with space: "Pre Fall 2026"
    {
        "label": "FASHION_SEASON",
        "pattern": [
            {"LOWER": "pre"},
            {"LOWER": "fall"},
            {"TEXT": {"REGEX": r"^\d{2,4}$"}}
        ]
    },

    # Pre-fall tokenized as three tokens: "Pre / Fall 2026"
    {
        "label": "FASHION_SEASON",
        "pattern": [
            {"LOWER": "pre"},
            {"TEXT": "-"},
            {"LOWER": "fall"},
            {"TEXT": {"REGEX": r"^\d{2,4}$"}}
        ]
    },

    # --------------------- Full Season Names ---------------------
    # Variations with full season name using separator: "Spring/Summer 2026", "Autumn/Winter 2026", "Fall-Winter 2026"
    {
        "label": "FASHION_SEASON",
        "pattern": [
            {"LOWER": {"IN": ["spring", "summer", "autumn", "fall", "winter"]}}, # should i remove sm and w??
            {"TEXT": {"IN": ["/", "-"]}},
            {"LOWER": {"IN": ["summer", "winter"]}},
            {"TEXT": {"REGEX": r"^\d{2,4}$"}}
        ]
    },

    # Variations with full season name, no separator: "Spring Summer 2026", "Autumn Winter 2026", "Fall Winter 2026"
    {
        "label": "FASHION_SEASON",
        "pattern": [
            {"LOWER": {"IN": ["spring", "summer", "autumn", "fall", "winter"]}},
            {"LOWER": {"IN": ["summer", "winter"]}},
            {"TEXT": {"REGEX": r"^\d{2,4}$"}}
        ]
    },
    
    # Variations with one single season: "Spring 2026", "Autumn 2026", "Fall 2026"
    {
        "label": "FASHION_SEASON",
        "pattern": [
            {"LOWER": {"IN": ["spring", "summer", "autumn", "fall", "winter"]}},
            {"TEXT": {"REGEX": r"^\d{2,4}$"}}
        ]
    },

    # --------------------- Year First Format ---------------------  
    # Variations in the order year and two season name: "2026 Spring/Summer"
    {
        "label": "FASHION_SEASON",
        "pattern": [
            {"TEXT": {"REGEX": r"^\d{2,4}$"}},
            {"LOWER": {"IN": ["spring", "summer", "autumn", "fall", "winter"]}},
            {"TEXT": {"IN": ["/", "-"]}},
            {"LOWER": {"IN": ["summer", "winter"]}}
        ]
    },

    # Variations in the order year and single season name: "2026 Spring", "2026 Autumn", "2026 Winter"
    {
        "label": "FASHION_SEASON",
        "pattern": [
            {"TEXT": {"REGEX": r"^\d{2,4}$"}},
            {"LOWER": {"IN": ["spring", "summer", "autumn", "fall", "winter"]}}
        ]
    },

    {
        "label": "FASHION_SEASON",
        "pattern": [
            {"TEXT": {"REGEX": r"^\d{2,4}$"}},
            {"LOWER": {"IN": ["resort", "cruise"]}},
        ]
    },

    {
        "label": "FASHION_SEASON",
        "pattern": [
            {"TEXT": {"REGEX": r"^\d{2,4}$"}},
            {"LOWER": {"IN": "pre"}},
            {"TEXT": {"IN": "/"}},
            {"LOWER": {"IN": "fall"}}
        ]
    },

    # --------------------- Collection Types ---------------------  
    {"label": "COLLECTION_TYPE", "pattern": "Haute Couture"},
    {"label": "COLLECTION_TYPE", "pattern": "haute couture"},
    {"label": "COLLECTION_TYPE", "pattern": "Couture"},
    {"label": "COLLECTION_TYPE", "pattern": "couture"},
    {"label": "COLLECTION_TYPE", "pattern": "Ready-to-Wear"},
    {"label": "COLLECTION_TYPE", "pattern": "ready-to-wear"},
    {"label": "COLLECTION_TYPE", "pattern": "Womenswear"},
    {"label": "COLLECTION_TYPE", "pattern": "womenswear"},
    {"label": "COLLECTION_TYPE", "pattern": "Menswear"},
    {"label": "COLLECTION_TYPE", "pattern": "menswear"}
]

ruler.add_patterns(fashion_season_patterns)

In [6]:
# TODO: TEST CASES > Write test cases to show that the patterns above are accurately identifying the seasons regardless of how they are written 
# This proves you know how sw development is supposed to be done (testing and more testing). 

In [7]:
# TODO: Normalize seasons. Ideal to eventually normalize fashion seasons into a common format.

<!-- # Comprehensive test cases
# test_cases = [
#     # Abbreviated
#     ("SS26 collection", "SS26"),
#     ("S/S26 trends", "S/S26"),
#     ("The AW2026 lookbook", "AW2026"),
#     ("New F/W 26 pieces", "F/W 26"),
    
#     # Pre-collections
#     ("Resort 2026 preview", "Resort 2026"),
#     ("Pre-Fall 2026 drops soon", "Pre-Fall 2026"),
#     ("Cruise 26 capsule", "Cruise 26"),
    
#     # Full season names
#     ("Spring/Summer 2026 runway", "Spring/Summer 2026"),
#     ("Fall-Winter 2026 trends", "Fall-Winter 2026"),
#     ("Spring 2026 ready-to-wear", "Spring 2026"),
    
#     # Year first
#     ("2026 Spring collection", "2026 Spring"),
#     ("2026 Fall/Winter preview", "2026 Fall/Winter"),
    
#     # Should NOT match
#     ("next spring trends", None),
#     ("summer vibes", None),
# ]

# print("TESTING SEASON EXTRACTION:\n")
# for text, expected in test_cases:
#     doc = nlp(text)
#     matches = [ent.text for ent in doc.ents if ent.label_ == "FASHION_SEASON"]
    
#     match_found = matches[0] if matches else None
#     status = "✅" if match_found == expected else "❌"
    
#     print(f"{status} '{text:35}' → Found: {match_found} (Expected: {expected})") -->

In [8]:
from collections import Counter
# Testing our season extraction pattern on collected articles
# TEST 1: Load real articles
# TEST 2: Extract seasons from articles
# TEST 3: Analyze results
# TEST 4: Look at the actual extractions
# TEST 5: Find false positives
# We did this above, are we doing it again???
# --------------------- Load Articles ---------------------
def load_articles(limit = 4100):
    """Loading articles from database"""
    query = """
        SELECT id, source_name, title, content, published_at
        FROM aggregates_trend
        WHERE published_at > NOW() - INTERVAL '365 days'
        ORDER BY published_at DESC
        LIMIT %s
    """

    with get_connection() as conn:
        data_frame = pd.read_sql(query, conn, params = (limit,))

    data_frame['full_text'] = data_frame['title'] + ' ' + data_frame['content'].fillna('')
        
    return data_frame
    
data_frame = load_articles() 

print(f"Loaded {len(data_frame)} articles.")
print(f"Published dates ranging from {data_frame['published_at'].min()} to {data_frame['published_at'].max()}") # Normalize the dates into Mon / DD / YYYY HH12:MI:SS
# min_date.strftime("%m-%d-%Y %I:%M:%S %p")

# TASK 1: What seasons are being mentioned? SPRING, SUMMER, AUTUMN, WINTER
# --------------------- Extract Seasons ---------------------
def extract_seasons(data_frame):
    """Extract the seasons mentioned per article."""
    seasons_found = []
    # seasons_found_dict = set() # TASK 1 I do think a set would look cleaner instead of having a long array with duplicate values
    articles_with_season_or_collection_mentions = []

    for i, row in data_frame.iterrows():
        # text = f"{row['title']} {row['content'] or ''}"
        text = row['full_text']   
        doc = nlp(text)
        # print(f"{row['title']} : {row['content']}")
        collection_mentioned = []
        season_mentioned = []

        for ent in doc.ents:
            if ent.label_ == "FASHION_SEASON":
                seasons_found.append(ent.text)
                season_mentioned.append(ent.text) 
            if ent.label_ == "COLLECTION_TYPE":
                collection_mentioned.append(ent.text)

        # if season_mentioned or collection_mentioned: # AND or OR
        #     articles_with_season_or_collection_mentions.append({
        #         'source': row['source_name'],
        #         'title': row['title'],
        #         'content': row['content'],
        #         'seasons': season_mentioned,
        #         'collection': collection_mentioned
        #     })


    # --------------------- Analyze Results ---------------------
    # TASK 2: How many articles mention a season? Which articles mention seasons? (append these articles into this list)
    season_mentions = Counter(seasons_found)
    print(f"Seasons mentioned: {season_mentions}")

    for seasons, count in season_mentions.most_common(100):
        label = "mention" if count <= 1 else "mentions"
        print(f"{seasons:25} {count:2} {label}")
        
    # For Debugging
    print(f"Total articles analyzed: {len(data_frame)}")
    print(f"Total articles with season/collection mentions: {len(articles_with_season_or_collection_mentions)} ({len(articles_with_season_or_collection_mentions)/len(data_frame)*100:.2f}%)")
    print(f"Total season mentions: {sum(season_mentions.values())}")


    # --------------------- WORKING ON BRAND + FASHION_SEASON ---------------------
    known_brands = [
        "Abercrombie", "Adidas", "Alexander McQueen", "Alexander Wang", "Amina Muaddi", "Asics", "Azzedine Alaïa",
        "Badgley Mischka", "Balenciaga", "Balmain", "Bottega Veneta", "Brunello Cucinelli", "Bulgari", "Burberry",
        "Calvin Klein", "Carolina Herrera", "Cartier", "Casadei", "Celia Kritharioti", "Celine", "Chanel", "Chloé", "Christian Dior", "Christian Lacroix", "Christian Louboutin", "Christopher Esber", "Chrome Hearts", "Coach",
        "DÔEN", "Dolce & Gabbana", "Dsquared2",
        "Elie Saab", "Emilio Pucci",
        "Fendi",
        "Ganni", "Gianvito Rossi", "Giorgio Armani", "Giuseppe Zanotti", "Givenchy", "Gucci",
        "H&M", "Helmut Lang", "Hermès", "Herve Leger",
        "Isabel Marant",
        "Jacquemus", "Jean Paul Gaultier", "Jill Sander", "Jimmy Choo", "John Galliano",
        "Khaite",
        "Lanvin", "Levi's", "Loewe", "Louis Vuitton", "Luar",
        "Maison Margiela", "Mango", "Manolo Blahnik", "Marc Jacobs", "MaxMara", "Michael Kors", "Miss Sohee", "Miu Miu", "Moschino",
        "New Balance", "Nike", "North Face",
        "Off-White", "Oscar de la Renta",
        "Paco Rabanne", "Prada", "Puma",
        "Ralph Lauren", "Robert Wun", "Roberto Cavalli",
        "Salomon", "Salvatore Ferragamo", "Schiaparelli", "Skims", "Stella McCartney", "Steve Madden",
        "The Attico", "The Row", "Thierry Mugler", "Thom Browne", "Tom Ford", "Toteme",
        "Valentino", "Van Cleef & Arpels", "Vera Wang", "Versace", "Vetements", "Victoria Beckham", "Vivienne Westwood",
        "Yohji Yamamoto", "Yves Saint Laurent",
        "Zac Posen", "Zara", "Zimmermann", "Zuhair Murad"
    ]
    
    # ================================================================================================================
    # TO-DO: Create aliases for the brands
    # ================================================================================================================

    brand_patterns = [
        {"label": "FASHION_BRAND", "pattern": brand.lower()}
        for brand in known_brands
    ]
    
    ruler.add_patterns(brand_patterns)
    
    brands_mentioned = [] # set

    for i, row in data_frame.iterrows():
        text = row['full_text']   
        doc = nlp(text)

        for ent in doc.ents:
            if ent.label_ == "FASHION_BRAND":
                # print(f"Found one brand: {ent.text} ({ent.label_})")
                brands_mentioned.append(ent.text)

    # Brand mention count
    brand_count = Counter(brands_mentioned)
    # print(brand_count)

    print("*" * 70)
    for brand, count in brand_count.most_common(100):
        label = "mention" if count <= 1 else "mentions"
        print(f"{brand:25} {count:2} {label}")

    # --------------------- WORKING ON MATERIAL'S MENTIONED ---------------------
    # Removed "down", "jersey", because of context (new 2/16 felt is also being mis...diagnosed no lol. mistaken, idk)
    materials = [
        "acetate", "acrylic", "alpaca",
        "bamboo", "bio-based", "boucle", "bouclé",
        "cactus leather", "canvas", "cashmere", "cashmere wool", "cellulosic fibres", "cellulosic fibers", "chambray", "chenille", "chiffon", "chitosan", "corduroy", "cotton", "crepe", "crêpe", "cupro",
        "damask", "denim", "ecovero", "elastane", "elastene", "elastin",
        "faux fur", "felt", "flannel", "flax", "fleece", "french terry", "fur",
        "georgette", "gingham", 
        "hemp",
        "jacquard", "jute",
        "lace", "latex", "leather", "linen", "lyocell",
        "merino", "merino wool", "mesh", "microfiber", "microfibre", "modal", "mohair", "muslin", "mycelium",
        "neoprene", "nylon",
        "organza", "oxford",
        "pinatex", "piñatex",
        "pima cotton", "pique", "piqué", "polyamide", "polyester", "polyvinyl chloride", "poplin", "pvc",
        "ramie", "rayon", "recycled cotton", "recycled nylon", "recycled pet", "recycled polyester", "ripstop",
        "satin", "seacell", "seersucker", "silk", "spandex", "spinnova", "suede",
        "taffeta", "tencel", "terry cloth", "toile", "toile de jouy", "tulle", "tweed", "twill", "tyvek",
        "velvet", "vinylon", "viscose", "viscose rayon", "voile",
        "waffle knit", "wool"
    ]

    material_pattern = [
        {"label": "MATERIAL", "pattern": material}
        for material in materials
    ]
    
    ruler.add_patterns(material_pattern)
    
    materials_mentioned = []

    for i, row in data_frame.iterrows():
        text = row['full_text']   
        doc = nlp(text)

        for ent in doc.ents:
            if ent.label_ == "MATERIAL":
                # print(text)
                # print(f"Found one material: {ent.text} ({ent.label_})")
                materials_mentioned.append(ent.text)

    matlist=Counter(materials_mentioned)
    # print(matlist)
    
    print("*" * 70)
    for mat, count in matlist.most_common(100):
        label = "mention" if count <= 1 else "mentions"
        print(f"{mat:25} {count:2} {label}")
# Runner code (notice no indent) Added this up here
extract_seasons(data_frame)

/var/folders/sc/ncg3j31n7j7fzg03p443n1mh0000gn/T/ipykernel_27081/1559715279.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, conn, params = (limit,))


Loaded 4100 articles.
Published dates ranging from 2025-10-03 05:00:00 to 2026-03-30 11:15:10
Seasons mentioned: Counter({'fall 2026': 173, 'spring 2026': 61, 'pre-fall 2026': 16, 'fall/winter 2026': 13, 'resort 2026': 8, '2026 winter': 7, 'summer 2026': 6, 'spring/summer 2026': 5, 'fall 2025': 5, 'winter 2025': 5, 'fw26': 3, 'fall/winter 2025': 3, 'fall 1995': 2, 'cruise 2026': 2, 'spring 2023': 1, 'winter 2026': 1, 'fall 2027': 1, 'spring 2006': 1, '22 winter': 1, 'spring 2024': 1, 'fall 1996': 1, 'resort 2027': 1, 'cruise 2027': 1, '28 winter': 1, 'fall 2006': 1, '14 winter': 1, 'spring 2025': 1, 'ss26': 1})
fall 2026                 173 mentions
spring 2026               61 mentions
pre-fall 2026             16 mentions
fall/winter 2026          13 mentions
resort 2026                8 mentions
2026 winter                7 mentions
summer 2026                6 mentions
spring/summer 2026         5 mentions
fall 2025                  5 mentions
winter 2025                5 mentions


In [9]:
# --------------------- TREND ANALYSIS ---------------------
def fashion_vocab(limit = 4000):
    """Loading articles from database"""
    query = """
        SELECT id, source_name, title, content, published_at
        FROM aggregates_trend
        WHERE published_at > NOW() - INTERVAL '3650 days'
        ORDER BY published_at DESC 
        LIMIT %s
    """

    with get_connection() as conn:
        data_frame = pd.read_sql(query, conn, params = (limit,))

    data_frame['full_text'] = data_frame['title'] + ' ' + data_frame['content'].fillna('')
        
    # I already declared this elsewhere. This will be deleted once we put it all together
    materials = [
        "acetate", "acrylic", "alpaca",
        "bamboo", "bio-based", "boucle", "bouclé",
        "cactus leather", "canvas", "cashmere", "cashmere wool", "cellulosic fibres", "cellulosic fibers", "chambray", "chenille", "chiffon", "chitosan", "corduroy", "cotton", "crepe", "crystal", "crêpe", "cupro",
        "damask", "denim", "ecovero", "elastane", "elastene", "elastin",
        "faux fur", "felt", "flannel", "flax", "fleece", "french terry", "fur",
        "georgette", "gingham", "hemp", "jacquard", "jute",
        "lace", "latex", "leather", "linen", "lyocell",
        "merino", "merino wool", "mesh", "microfiber", "microfibre", "modal", "mohair", "muslin", "mycelium",
        "neoprene", "nylon", "organza", "oxford", "pinatex", "piñatex",
        "pima cotton", "pique", "piqué", "polyamide", "polyester", "polyvinyl chloride", "poplin", "pvc",
        "ramie", "rayon", "recycled cotton", "recycled nylon", "recycled pet", "recycled polyester", "ripstop",
        "satin", "seacell", "seersucker", "silk", "spandex", "spinnova", "suede",
        "taffeta", "tencel", "terry cloth", "toile", "toile de jouy", "tulle", "tweed", "twill", "tyvek",
        "velvet", "vinylon", "viscose", "viscose rayon", "voile",
        "waffle knit", "wool"
    ]

    known_brands = [
        "Abercrombie", "Adidas", "Alexander McQueen", "Alexander Wang", "Amina Muaddi", "Aritzia", "Asics", "Azzedine Alaïa",
        "Badgley Mischka", "Balenciaga", "Balmain", "Bottega Veneta", "Brunello Cucinelli", "Bulgari", "Burberry",
        "Calvin Klein", "Carolina Herrera", "Cartier", "Casadei", "Celia Kritharioti", "Celine", "Chanel", "Chloé", "Christian Dior", "Christian Lacroix", "Christian Louboutin", "Christopher Esber", "Chrome Hearts", "Coach",
        "DÔEN", "Dolce & Gabbana", "Dsquared2",
        "Elie Saab", "Emilio Pucci",
        "Fendi",
        "Ganni", "Gianvito Rossi", "Giorgio Armani", "Giuseppe Zanotti", "Givenchy", "Gucci",
        "H&M", "Helmut Lang", "Hermès", "Herve Leger",
        "Isabel Marant",
        "Jacquemus", "Jean Paul Gaultier", "Jill Sander", "Jimmy Choo", "John Galliano",
        "Khaite",
        "Lanvin", "Levi's", "Loewe", "Louis Vuitton", "Luar",
        "Maison Margiela", "Mango", "Manolo Blahnik", "Marc Jacobs", "MaxMara", "Michael Kors", "Miss Sohee", "Miu Miu", "Moschino",
        "New Balance", "Nike", "North Face",
        "Off-White",  "Oscar de la Renta",
        "Paco Rabanne", "Prada", "Puma",
        "Ralph Lauren", "Robert Wun", "Roberto Cavalli",
        "Salomon", "Salvatore Ferragamo", "Schiaparelli", "Skims", "Stella McCartney", "Steve Madden",
        "The Attico", "The Row", "Thierry Mugler", "Thom Browne", "Tom Ford", "Toteme", "UGG",
        "Valentino", "Van Cleef & Arpels", "Vera Wang", "Versace", "Vetements", "Victoria Beckham", "Vivienne Westwood",
        "Yohji Yamamoto", "Yves Saint Laurent",
        "Zac Posen", "Zara", "Zimmermann", "Zuhair Murad"
    ]

    fashion_nouns = [
        "accessory", "bag", "bralette", "corset", "corsetry", "hat", "jewelry", "watch", "earring", "top", "pant",
        "gown", "dress", "necklace", "sandal", "jacket", "jean", "knitwear", "skirt", "sweater", "tee", "coat", "cape",
        "boot", "flat", "heel", "lingerie", "loafer", "outerwear", "sandal", "sneaker", "suit", "top", "undergarment"
    ]
    
    fashion_adjectives = [
        "backless", "baggy", "ballet", "barrel", "basic", "bespoke", "bodycon", "bomber", "cable",
        "colorful", "cowboy", "cozy", "crop", "cutout", "drape", "duster", "embellish", "embroider", "fine", "fit", "fringe", "full", "halter",
        "hand-crafted", "jeweled", "knee-length", "knit", "lightweight", "loose", "maxi", "metallic", "midi",
        "mini", "miniskirt", "minimalist", "naked", "over-the-knee", "oversize", "penny", "plunge", 
        "polish", "polo", "relax", "semi-sheer", "sheer", "shimmer", "sleeveless", "slim", "strapless", "structure",  "tailor", "tailored", "tall", "trim", "weave"
    ]

    matcher = Matcher(nlp.vocab)
 
    # Pattern catches: adjective + noun (oversized blazer)
    adj_noun = [
        {"LEMMA": {"IN": [fashion_adj.lower() for fashion_adj in fashion_adjectives]}},
        {"LEMMA": {"IN": [fashion_noun.lower() for fashion_noun in fashion_nouns]}}
    ]

    # Pattern catches: material + noun (leather jacket)
    mat_noun = [
        {"LEMMA": {"IN": [material.lower() for material in materials]}},
        {"LEMMA": {"IN": [fashion_noun.lower() for fashion_noun in fashion_nouns]}}
    ]

    # Pattern catches: material + adjective + noun (oversized leather jacket)
    mat_adj_noun = [
        {"LEMMA": {"IN": [fashion_adj.lower() for fashion_adj in fashion_adjectives]}},
        {"LEMMA": {"IN": [material.lower() for material in materials]}},
        {"LEMMA": {"IN": [fashion_noun.lower() for fashion_noun in fashion_nouns]}}
    ]

    # Pattern catches: brand name + noun (Gucci loafers)
    brand_noun = [
        {"LOWER": {"IN": [brand.lower() for brand in known_brands]}},
        {"LEMMA": {"IN": [fashion_noun.lower() for fashion_noun in fashion_nouns]}}
    ]

    # Pattern catches: brand name + adjective + noun (Gucci penny loafers)
    brand_adj_noun = [
        {"LOWER": {"IN": [brand.lower() for brand in known_brands]}},
        {"LEMMA": {"IN": [fashion_adj.lower() for fashion_adj in fashion_adjectives]}},
        {"LEMMA": {"IN": [fashion_noun.lower() for fashion_noun in fashion_nouns]}}
    ]

    matcher.add("FASHION_TREND", [adj_noun, mat_noun, mat_adj_noun, brand_noun, brand_adj_noun])

    all_found_trends = []
    for i, row in data_frame.iterrows():
        doc = nlp(row['full_text'])
        matches = matcher(doc)
        
        found_trends = set() # Deduplicating the trends (if article mentions > 1, count = 1)

        for match_id, start, end in matches:
            span = doc[start:end]
            trend_lemma = span.lemma_
            trend_raw = span.text
            print(f"  TREND (lemma): '{trend_lemma}'")
            print(f"  TREND (raw): '{trend_raw}'")

            # We're storing both lemma & raw text. Lemma for counting, raw for displaying.
            found_trends.add((trend_lemma, trend_raw)) 
            
        # Add found trends in our set to all the found trends list
        for trend in found_trends:
            all_found_trends.append(trend)


    trend_dic = {} # Add them to a dictionary
    for trend_lemma, trend_raw in all_found_trends:
        if trend_lemma not in trend_dic:
            trend_dic[trend_lemma] = trend_raw

    trend_counts = Counter(trend_lemma for trend_lemma, trend_raw in all_found_trends) # Using the set to count
    
# ----
    # Print results
    print(f"\nAnalyzed {len(data_frame)} articles")
    print(f"Found {len(all_found_trends)} total trend mentions") # Total number of mentions
    print(f"Found {len(trend_counts)} unique trends (this is using the counter)\n") # Total without duplicates... I think.
    
    print("=" * 60)
    print("TOP 150 FASHION TRENDS")
    print("=" * 60)
    
    # for trend, count in trend_counts.most_common(200):
    #     print(f"{trend:30} {count:>3} mentions")
    # ----
    
    for trend_lemma, count in trend_counts.most_common(150):
        display = trend_dic[trend_lemma]
        print(f"{display:30} {count:>3} mentions")

    return data_frame
    
data_frame = fashion_vocab()

/var/folders/sc/ncg3j31n7j7fzg03p443n1mh0000gn/T/ipykernel_27081/3622125946.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, conn, params = (limit,))


  TREND (lemma): 'cutout dress'
  TREND (raw): 'cutout dress'
  TREND (lemma): 'leather suit'
  TREND (raw): 'leather suit'
  TREND (lemma): 'lightweight jacket'
  TREND (raw): 'lightweight jackets'
  TREND (lemma): 'colorful jean'
  TREND (raw): 'colorful jeans'
  TREND (lemma): 'minimalist gown'
  TREND (raw): 'minimalist gown'
  TREND (lemma): 'chanel dress'
  TREND (raw): 'chanel dress'
  TREND (lemma): 'chanel gown'
  TREND (raw): 'chanel gown'
  TREND (lemma): 'strapless gown'
  TREND (raw): 'strapless gown'
  TREND (lemma): 'naked dress'
  TREND (raw): 'naked dresses'
  TREND (lemma): 'chanel fine jewelry'
  TREND (raw): 'chanel fine jewelry'
  TREND (lemma): 'fine jewelry'
  TREND (raw): 'fine jewelry'
  TREND (lemma): 'gucci gown'
  TREND (raw): 'gucci gown'
  TREND (lemma): 'givenchy suit'
  TREND (raw): 'givenchy suit'
  TREND (lemma): 'sheer skirt'
  TREND (raw): 'sheer skirt'
  TREND (lemma): 'suede jacket'
  TREND (raw): 'suede jacket'
  TREND (lemma): 'fine jewelry'
  TR

In [10]:
# --------------------- TREND ANALYSIS (old) ---------------------
# These are LINGUISTIC PATTERNS, not known terms. They're unknown combinations, therefore we need the matcher.
matcher = Matcher(nlp.vocab)

# adjective + (clothing) noun i.e., ruffled skirt
# noun + (clothing) noun i.e., zebra jacket
# adjective + adjective + noun i.e., bright yellow jacket
# noun + noun + noun i.e., faux leather jacket
trend_patterns = [
    [{"POS": "NOUN"}, {"POS": "ADJ"}],
    [{"POS": "ADJ"}, {"POS": "NOUN"}],
    [{"POS": "ADJ"}, {"POS": "NOUN"}, {"POS": "NOUN"}],
    [{"POS": "NOUN"}, {"POS": "NOUN"}],
    [{"POS": "ADJ"}, {"POS": "ADJ"}]
]

current_count = 0

matcher.add("LINGUISTIC_PATTERN", trend_patterns)
for index, row in data_frame.iterrows():
    text = row['full_text']   
    # doc = nlp(text)
    # doc = nlp("bottega veneta’s ceo to exit and take helm at moncler leo rongone kept up momentum at the italian leather goods house through two designer transitions, onboarding matthieu blazy and louise trotter. now he'll take on the role of group ceo at moncler.")
    doc=nlp("kpop demon hunters’ rei ami, ejae, and audrey nuna put fresh spins on black evening gowns the trio, who are the singing voices of huntr/x, made their critics choice awards debut in style.")

    matches = matcher(doc)
    # print(text)
    for match_id, start, end in matches:
        print(nlp.vocab.strings[match_id], "->", doc[start:end])
    # print("Total matches found:", len(matches))
    print("*" * 50)

    # print("Current matches found:", counttt)
    
# Create POS tagging with a fashion vocabulary filter. Where to get the fashion vocabulary?
# Hard-coded? List? Database?
# Combine POS patterns with a fashion vocabulary
# trends_mentioned = []

# For every entry/row/article in our database, if we encounter a pattern such as  NOUN + NOUN, ADJ + NOUN append it to our trends list so that we can see what trends we are seeing
# for index, row in data_frame.iterrows():
#     text = row['full_text']   
#     doc = nlp(text)
#     matches = matcher(doc)

#     # print("Total matches found:", len(matches))
#     for match_id, start, end in matches:
#         span = doc[start:end]
#         print(span.text)


# # Add the pattern to the matcher and apply the matcher to the doc
# matcher.add("ADJ_NOUN_PATTERN", [pattern])
# matches = matcher(doc)
# print("Total matches found:", len(matches))
# for match_id, start, end in matches:
#     print("Match found:", doc[start:end].text)

# def get_fashion_nouns(articles_df):
#     """Extract all nouns from articles (potential trend keywords)."""
#     all_nouns = []
    
#     for text in articles_df['full_text']:
#         doc = nlp(text.lower())
#         nouns = [token.lemma_ for token in doc if token.pos_ == 'NOUN']
#         all_nouns.extend(nouns)
    
#     return Counter(all_nouns)

# # Run on your data
# fashion_nouns = get_fashion_nouns(data_frame)
# print(fashion_nouns.most_common(10))



LINGUISTIC_PATTERN -> fresh spins
LINGUISTIC_PATTERN -> black evening
LINGUISTIC_PATTERN -> black evening gowns
LINGUISTIC_PATTERN -> evening gowns
LINGUISTIC_PATTERN -> singing voices
LINGUISTIC_PATTERN -> critics choice
LINGUISTIC_PATTERN -> choice awards
LINGUISTIC_PATTERN -> awards debut
**************************************************
LINGUISTIC_PATTERN -> fresh spins
LINGUISTIC_PATTERN -> black evening
LINGUISTIC_PATTERN -> black evening gowns
LINGUISTIC_PATTERN -> evening gowns
LINGUISTIC_PATTERN -> singing voices
LINGUISTIC_PATTERN -> critics choice
LINGUISTIC_PATTERN -> choice awards
LINGUISTIC_PATTERN -> awards debut
**************************************************
LINGUISTIC_PATTERN -> fresh spins
LINGUISTIC_PATTERN -> black evening
LINGUISTIC_PATTERN -> black evening gowns
LINGUISTIC_PATTERN -> evening gowns
LINGUISTIC_PATTERN -> singing voices
LINGUISTIC_PATTERN -> critics choice
LINGUISTIC_PATTERN -> choice awards
LINGUISTIC_PATTERN -> awards debut
*******************

In [11]:
# Runner code (notice no indent)
extract_seasons(data_frame)

Seasons mentioned: Counter({'fall 2026': 173, 'spring 2026': 60, 'pre-fall 2026': 16, 'fall/winter 2026': 13, 'resort 2026': 8, '2026 winter': 7, 'summer 2026': 6, 'spring/summer 2026': 5, 'fall 2025': 5, 'winter 2025': 5, 'fw26': 3, 'fall/winter 2025': 3, 'fall 1995': 2, 'cruise 2026': 2, 'spring 2023': 1, 'winter 2026': 1, 'fall 2027': 1, 'spring 2006': 1, '22 winter': 1, 'spring 2024': 1, 'fall 1996': 1, 'resort 2027': 1, 'cruise 2027': 1, '28 winter': 1, 'fall 2006': 1, '14 winter': 1, 'spring 2025': 1, 'ss26': 1})
fall 2026                 173 mentions
spring 2026               60 mentions
pre-fall 2026             16 mentions
fall/winter 2026          13 mentions
resort 2026                8 mentions
2026 winter                7 mentions
summer 2026                6 mentions
spring/summer 2026         5 mentions
fall 2025                  5 mentions
winter 2025                5 mentions
fw26                       3 mentions
fall/winter 2025           3 mentions
fall 1995         

In [12]:
# We need to train a NER model to be able to distinguish between a person and brand, costly. Do after launch of V0 using open source tools.

In [13]:
# OLD WORK Matcher is if we want to match...? Certain spans?
# We began by trying to identify eras. Good thinking, shouldn't be that difficult but not a priority rn.
# Write custom fashion matching patterns. Models are statistical and not always right. 
# Whether their predictions are correct depends on the training data and the text you’re processing.

# # TO-DO Create a matching pattern.
# matcher = Matcher(nlp.vocab)

# # Create a custom entity ruler for fashion eras
# entity_ruler = nlp.add_pipe("entity_ruler")

# # Ensuring important fashion eras in the last century are recognized
# eras = [{
#     "label": "ERA",
#     "pattern": [
#         {"LOWER": {"REGEX":r"^(18|19|20)\d0s$|^(0?\d{1,2})s$|^y2k$"}}
#     ]
# }]

# # Add the eras pattern to the entity ruler
# entity_ruler.add_patterns(eras)
